
# 🐼 EDA con Pandas — comparación con NumPy

En esta práctica repetimos el **mismo análisis exploratorio (EDA)** que hicimos con NumPy,  
pero ahora usando **Pandas**, para ver cómo cambia la sintaxis, el tiempo de desarrollo  
y las capacidades de análisis.

---

## 🧾 Preparación del dataset

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

df = pd.DataFrame({
    "ID": np.arange(1, n+1),
    "Edad": rng.integers(18, 65, size=n),
    "Salario": rng.normal(2500, 800, size=n),
    "Experiencia": np.clip(rng.integers(0, 40, size=n), 0, None),
    "Horas": rng.normal(40, 5, size=n),
    "Nivel": rng.integers(0, 5, size=n),
    "Satisfacción": rng.normal(7, 2, size=n)
})

# Añadimos outliers y NaN
df.loc[rng.choice(n, 10, replace=False), "Salario"] *= 4
df.loc[rng.choice(n, 15, replace=False), "Satisfacción"] = np.nan

df_original = df.copy(deep=True)  # conservar los datos antes de limpiarlos
df.head(10)

,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción
0,1,22,3646.571605,13,30.534144,1,4.369942
1,2,54,2573.216143,21,41.086366,3,4.237672
2,3,48,2964.621676,7,47.875414,1,7.982905
3,4,38,2454.573444,35,42.443343,4,11.127789
4,5,38,2363.673935,29,40.853125,0,9.724916
5,6,58,1876.414083,30,37.197753,2,3.208030
6,7,22,2844.241087,6,40.155236,1,9.842152
7,8,50,1818.770249,28,38.739686,3,8.872801
8,9,27,3032.468189,33,36.070117,3,3.909400
9,10,22,3368.229604,7,38.550622,4,6.393597


---

## 🧠 Tarea 1: Descripción básica

In [2]:
print(df.shape)
print(df.columns)
df.head()

(1000, 7)
Index(['ID', 'Edad', 'Salario', 'Experiencia', 'Horas', 'Nivel',
       'Satisfacción'],
      dtype='object')


,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción
0,1,22,3646.571605,13,30.534144,1,4.369942
1,2,54,2573.216143,21,41.086366,3,4.237672
2,3,48,2964.621676,7,47.875414,1,7.982905
3,4,38,2454.573444,35,42.443343,4,11.127789
4,5,38,2363.673935,29,40.853125,0,9.724916


✅ **Más directo**: Pandas ya muestra etiquetas y formato de tabla.

---

## 📏 Tarea 2: Tipos y rangos

In [3]:
print(df.dtypes)
df[["Edad", "Salario"]].agg(["min", "max", "mean", "median", "std"])

ID                int64
Edad              int64
Salario         float64
Experiencia       int64
Horas           float64
Nivel             int64
Satisfacción    float64
dtype: object


,Edad,Salario
min,18.000000,-418.730260
max,64.000000,14759.674527
mean,41.136000,2558.904976
median,41.000000,2509.158692
std,13.550022,1152.322209


✅ En una sola línea obtenemos estadísticas básicas.

---

## 💭 Tarea 3: Filtrado de datos

In [4]:
# 1. Salarios > 4000 €
print(df[df["Salario"] > 4000])

# 2. Más de 45h y >10 años experiencia
print(df[(df["Horas"] > 45) & (df["Experiencia"] > 10)][["ID", "Edad", "Horas", "Experiencia"]])

# 3. Porcentaje nivel ≥ 3
print((df["Nivel"] >= 3).mean() * 100)

      ID  Edad       Salario  Experiencia      Horas  Nivel  Satisfacción
25    26    48   4436.332010           28  36.670611      3      7.531057
36    37    58   7797.208697            8  43.293987      0      6.537106
72    73    39   4128.058642           14  40.511283      1      7.942501
86    87    22   7910.668337           20  41.364313      2      5.478271
104  105    55   4097.837225           25  42.036319      1      8.434697
150  151    58   4539.462362           26  51.032627      4      5.841120
161  162    49   4142.147986            5  44.231520      0      9.104150
195  196    22   4406.081826           39  34.612084      0     10.830869
209  210    54  11005.047116           29  40.086673      4      8.053396
272  273    25   4139.004521           16  39.726478      3      7.141989
302  303    18  12861.482651           21  37.267311      3      6.869909
359  360    53   4024.048171           36  40.423306      3      9.463959
365  366    42   5043.082943          

✅ Sintaxis más legible y declarativa.

---

## 🧹 Tarea 4: Limpieza — valores faltantes

In [6]:
print(df["Satisfacción"].isna().sum())
df["Satisfacción"] = df["Satisfacción"].fillna(df["Satisfacción"].mean())
print(df["Satisfacción"].isna().any())


0
False


✅ Un único método `fillna()` evita usar máscaras.

---

## ⚠️ Tarea 5: Detección de outliers

In [7]:
Q1 = df["Salario"].quantile(0.25)
Q3 = df["Salario"].quantile(0.75)
IQR = Q3 - Q1

filtro = (df["Salario"] < Q1 - 1.5*IQR) | (df["Salario"] > Q3 + 1.5*IQR)
df[filtro]

,ID,Edad,Salario,Experiencia,Horas,Nivel,Satisfacción
23,24,61,128.376930,22,51.070763,2,9.162750
36,37,58,7797.208697,8,43.293987,0,6.537106
86,87,22,7910.668337,20,41.364313,2,5.478271
209,210,54,11005.047116,29,40.086673,4,8.053396
302,303,18,12861.482651,21,37.267311,3,6.869909
348,349,26,361.732454,10,28.083056,4,7.535081
365,366,42,5043.082943,25,39.049642,3,7.462020
466,467,25,155.524499,2,35.141906,2,4.131556
468,469,26,-418.730260,38,44.096715,0,4.887469
499,500,38,11482.783379,23,40.182404,2,9.680509


✅ `quantile()` y condiciones combinadas simplifican la detección.

---

## 🧰 Tarea 6: Sustitución de outliers

In [8]:
# Ejemplo de winsorización: limitar ambos extremos sin elevar los salarios bajos.
p05, p95 = df["Salario"].quantile([0.05, 0.95])
df["Salario"] = df["Salario"].clip(lower=p05, upper=p95)


La winsorización limita los extremos a los percentiles 5 y 95. Es una decisión didáctica, no una regla universal: antes de tratar un valor extremo hay que comprobar si es un error o un caso real. Se conserva `df_original` para comparar.

---

## 📊 Tarea 7: Agrupaciones y resúmenes simples


In [9]:
# Media por nivel
print(df.groupby("Nivel")["Salario"].mean())
# Satisfacción media por grupo de edad
bins = [0, 30, 50, 100]
labels = ["≤30", "31–50", ">50"]
df["GrupoEdad"] = pd.cut(df["Edad"], bins=bins, labels=labels)
print(df.groupby("GrupoEdad", observed=True)["Satisfacción"].mean())

Nivel
0    2554.911303
1    2461.533403
2    2590.064240
3    2491.547201
4    2405.231405
Name: Salario, dtype: float64
GrupoEdad
≤30      7.021416
31–50    7.111431
>50      6.802487
Name: Satisfacción, dtype: float64


✅ `groupby()` y `cut()` hacen triviales las operaciones por grupo.

---

## 🧩 Tarea 8: Normalización y escalado

In [10]:
df["Salario_norm"] = (df["Salario"] - df["Salario"].min()) / (df["Salario"].max() - df["Salario"].min())
df["Horas_z"] = (df["Horas"] - df["Horas"].mean()) / df["Horas"].std()
df[["Salario_norm", "Horas_z"]].describe()

,Salario_norm,Horas_z
count,1000.000000,1.000000e+03
mean,0.498364,1.232792e-15
std,0.273043,1.000000e+00
min,0.000000,-3.060000e+00
25%,0.303496,-6.801900e-01
50%,0.500355,1.357540e-02
75%,0.684955,6.611005e-01
max,1.000000,3.005690e+00


✅ Con Pandas, las operaciones vectorizadas siguen siendo de NumPy,
pero la sintaxis es más limpia y las columnas se guardan fácilmente.

---

## 🧮 Tarea 9: Correlaciones simples

In [11]:
print(df[["Salario", "Experiencia"]].corr())
print(df[["Edad", "Satisfacción"]].corr())

              Salario  Experiencia
Salario      1.000000     0.015517
Experiencia  0.015517     1.000000
                 Edad  Satisfacción
Edad          1.00000      -0.03393
Satisfacción -0.03393       1.00000


✅ `corr()` calcula la matriz de correlación directamente.

---

## 🧾 Tarea 10: Estadísticas generales

In [ ]:
df.describe(include="all")

✅ Una sola línea genera resumen completo de medias, desvíos,
mínimos, máximos y cuantiles — con una interfaz tabular; NumPy también dispone de funciones de resumen.

---

## ⚖️ Comparación final: NumPy vs Pandas

| Aspecto                       | NumPy                                     | Pandas                                         |
| ----------------------------- | ----------------------------------------- | ---------------------------------------------- |
| **Estructura**                | Arrays de un mismo dtype; también texto y tipos estructurados          | DataFrames con columnas heterogéneas           |
| **Acceso a datos**            | Por índice numérico                       | Por nombre de columna                          |
| **Limpieza (NaN, outliers)**  | Manual, usando máscaras y `np.isnan()`    | Métodos directos (`fillna`, `replace`, `clip`) |
| **Agrupaciones**              | Manual, por filtrado repetido             | `groupby`, `cut`, `pivot_table`                |
| **Resúmenes**                 | Funciones separadas (`mean`, `std`, etc.) | `describe()`, `agg()` integrados               |
| **Eficiencia**                | Rendimiento según operación, dtype y datos                  | Rendimiento según operación, dtype y datos               |
| **Lectura de datos externos** | `loadtxt`, `genfromtxt`, `load` (entre otros) | CSV, Excel, SQL, Parquet, JSON…                |

---

## 🧠 Reflexión: cuándo usar cada uno

* 🧮 **NumPy**: ideal para **cálculos numéricos puros**, operaciones de álgebra, vectores, tensores o machine learning a bajo nivel.
* 🐼 **Pandas**: ideal para **análisis de datos reales** con valores faltantes, mezcla de tipos, y manipulación de columnas o tablas grandes.

En la práctica, **Pandas usa NumPy por debajo** para los cálculos numéricos,
pero aporta una capa de **semántica tabular y limpieza** que simplifica enormemente el trabajo.

---

## Tipos de datos anulables y PyArrow (ampliación opcional)

Pandas puede almacenar columnas con arrays NumPy, tipos anulables propios o tipos Arrow. Esto no equivale a cambiar un motor global de ejecución. No existe la opción `mode.dtype_backend`.

Para convertir una copia se usa `convert_dtypes(dtype_backend=...)` (pandas ≥ 2.0). La variante `pyarrow` requiere instalar PyArrow. No hace falta para el EDA de esta clase.

Referencia: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html


In [12]:
# Tipos anulables de pandas: no requieren PyArrow.
df_nullable = df.convert_dtypes()
print(df_nullable.dtypes)


ID                 Int64
Edad               Int64
Salario          Float64
Experiencia        Int64
Horas            Float64
Nivel              Int64
Satisfacción     Float64
GrupoEdad       category
Salario_norm     Float64
Horas_z          Float64
dtype: object


Conversión opcional a tipos Arrow, manteniendo el DataFrame de trabajo intacto:


In [13]:
try:
    import pyarrow
except ImportError:
    print("PyArrow no está instalado; continuamos con pandas sin esta ampliación.")
else:
    df_arrow = df.convert_dtypes(dtype_backend="pyarrow")
    print(df_arrow.dtypes)


ID               int64[pyarrow]
Edad             int64[pyarrow]
Salario         double[pyarrow]
Experiencia      int64[pyarrow]
Horas           double[pyarrow]
Nivel            int64[pyarrow]
Satisfacción    double[pyarrow]
GrupoEdad              category
Salario_norm    double[pyarrow]
Horas_z         double[pyarrow]
dtype: object


Arrow aporta tipos anulables e interoperabilidad con otras herramientas. Su consumo de memoria y rendimiento dependen de los datos y las operaciones: no siempre mejora a NumPy.

## Próximo paso

Compara la sintaxis y los resultados con el EDA de NumPy. Una semilla idéntica con generadores distintos no produce los mismos valores; para una comparación exacta hay que compartir los datos, el generador y el orden de las llamadas. NumPy usa por defecto `ddof=0` para la desviación típica; pandas usa `ddof=1`.

Después de introducir Matplotlib, continúa con `03_pandas_eda_visualizacion.ipynb`. Este cuaderno se centra en el análisis sin gráficas.
